# SleepPy Extraction Review

This notebook reviews first-pass sleep metric extraction from Oura Ring 4 finger, Oura Ring 3 toe, Samsung Watch / SleepWatch, Muse, and OSCAR/SleepScope sample screenshots or PDFs.

**Disclaimer:** This is exploratory wellness data analysis only. It is not medical diagnosis, treatment advice, or a replacement for clinician review.

## Workflow

1. Place 1-2 representative files per device under `data/raw/samples/<device>/`.
2. Run the extraction cell to create `data/processed/nightly_summary.csv`, `data/processed/device_observations_long.csv`, and `outputs/extraction_report.md`.
3. Review extracted values, missingness, confidence labels, and trend plots.
4. Correct low-confidence or missing values manually before treating them as analysis inputs.

In [9]:
from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import pandas as pd

# Reload local modules so an already-running notebook kernel picks up edits from disk.
import sleeppy.extract as sleep_extract
import sleeppy.extract.common as extract_common
import sleeppy.extract.pipeline as extract_pipeline
importlib.reload(extract_common)
importlib.reload(extract_pipeline)
importlib.reload(sleep_extract)

from sleeppy.extract.common import check_ocr_environment
from sleeppy.extract.pipeline import run_sample_extraction
from sleeppy.quality import confidence_by_device, missingness_by_device

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## OCR Diagnostics

The notebook must have dependencies installed in the active kernel shown below. If `image_ocr_ready` is `False`, run `import sys; !"{sys.executable}" -m pip install -r requirements.txt`, then restart the kernel.

In [10]:
ocr_status = check_ocr_environment()
display(pd.DataFrame([ocr_status]))

if not ocr_status["image_ocr_ready"]:
    print("Image OCR is not ready in this active notebook kernel. Screenshots will produce zero extracted values until this is fixed.")
    print(ocr_status["notes"])
else:
    print("Image OCR is ready.")

,python_executable,pillow_installed,pytesseract_installed,pymupdf_installed,tesseract_cmd,tesseract_version,image_ocr_ready,pdf_text_ready,notes
0,C:\Users\morri\miniconda3\python.exe,True,True,True,C:\Program Files\Tesseract-OCR\tesseract.exe,5.4.0.20240606,True,True,OCR/PDF dependencies look ready.


Image OCR is ready.


In [11]:
RUN_EXTRACTION = True

if RUN_EXTRACTION:
    nightly_summary, observations, report_path = run_sample_extraction(
        raw_samples_dir=PROJECT_ROOT / "data" / "raw" / "samples",
        processed_dir=PROCESSED_DIR,
        outputs_dir=OUTPUTS_DIR,
        include_legacy_raw=True,
    )
else:
    nightly_summary = pd.read_csv(PROCESSED_DIR / "nightly_summary.csv")
    observations = pd.read_csv(PROCESSED_DIR / "device_observations_long.csv")
    report_path = OUTPUTS_DIR / "extraction_report.md"

print(f"Night/device rows: {len(nightly_summary)}")
print(f"Extracted values: {len(observations)}")
print(f"Report: {report_path}")

Night/device rows: 1
Extracted values: 16
Report: C:\Users\morri\PycharmProjects\SleepPy\outputs\extraction_report.md


## Extracted Tables

In [12]:
display(nightly_summary)
display(observations.head(50))

,date,device,sleep_score,total_sleep_minutes,time_in_bed_minutes,sleep_efficiency_pct,awake_minutes,rem_minutes,light_minutes,deep_minutes,lowest_hr,avg_hr,avg_hrv,max_hrv,avg_spo2,respiratory_rate,breathing_label,cpap_mask_minutes,cpap_ahi,cpap_cai,cpap_oai,cpap_pressure_95,cpap_leak,source_files,extraction_methods,min_confidence,notes
0,2026-06-08,Samsung Watch / SleepWatch,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,57,<NA>,<NA>,<NA>,11.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,data\raw\samples\samsung_watch\Screenshot_2026...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...


,date,device,metric,value,unit,source_file,extraction_method,confidence,notes
0,NaT,Oura Ring 4 finger,awake_minutes,48,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
1,NaT,Oura Ring 4 finger,rem_minutes,73,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
2,NaT,Oura Ring 4 finger,light_minutes,265,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
3,NaT,Oura Ring 4 finger,deep_minutes,43,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
4,NaT,Oura Ring 4 finger,total_sleep_minutes,381,minutes,data\raw\samples\oura4\37A28C13-74A7-4059-B80D...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
5,NaT,Oura Ring 4 finger,sleep_efficiency_pct,89,pct,data\raw\samples\oura4\37A28C13-74A7-4059-B80D...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
6,NaT,Oura Ring 4 finger,lowest_hr,54,bpm,data\raw\samples\oura4\6B3E3B9B-1F0D-4EE7-B182...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
7,NaT,Oura Ring 4 finger,avg_hrv,20,ms,data\raw\samples\oura4\6B3E3B9B-1F0D-4EE7-B182...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
8,NaT,Oura Ring 4 finger,total_sleep_minutes,381,minutes,data\raw\samples\oura4\933CACBF-2408-46D2-923F...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
9,NaT,Oura Ring 4 finger,sleep_efficiency_pct,89,pct,data\raw\samples\oura4\933CACBF-2408-46D2-923F...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...


## Missingness and Confidence

In [13]:
missingness = missingness_by_device(nightly_summary)
confidence = confidence_by_device(observations)

if missingness.empty:
    print("No nightly summary rows available yet.")
else:
    display(missingness.pivot_table(index="metric", columns="device", values="missing_pct"))

if confidence.empty:
    print("No extracted observations available yet.")
else:
    display(confidence.pivot_table(index="device", columns="confidence", values="count", fill_value=0))

device,Samsung Watch / SleepWatch
metric,
avg_hr,0.0
avg_hrv,100.0
avg_spo2,100.0
awake_minutes,100.0
breathing_label,100.0
cpap_ahi,100.0
cpap_cai,100.0
cpap_leak,100.0
cpap_mask_minutes,100.0


confidence,low,medium
device,,
Oura Ring 4 finger,1.0,13.0
Samsung Watch / SleepWatch,0.0,2.0


## Trends

In [14]:
def plot_metric(summary_df, metric, ylabel, title):
    if summary_df.empty or metric not in summary_df:
        print(f"No {metric} data available.")
        return None

    df = summary_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    df = df.dropna(subset=["date", metric])
    if df.empty:
        print(f"No plottable {metric} values available.")
        return None

    fig, ax = plt.subplots(figsize=(10, 4))
    for device, group in df.groupby("device", sort=True):
        group = group.sort_values("date")
        ax.plot(group["date"], group[metric], marker="o", label=device)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Date")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig, ax

In [15]:
plot_metric(nightly_summary, "total_sleep_minutes", "Minutes", "Sleep duration")
plot_metric(nightly_summary, "avg_hrv", "ms", "Average HRV")
plot_metric(nightly_summary, "avg_spo2", "%", "Average SpO2")
plot_metric(nightly_summary, "cpap_ahi", "Events/hour", "CPAP AHI")
plt.show()

No plottable total_sleep_minutes values available.
No plottable avg_hrv values available.
No plottable avg_spo2 values available.
No plottable cpap_ahi values available.


## Extraction Report

In [16]:
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("No extraction report found yet.")

# SleepPy Extraction Report

This report summarizes automated first-pass extraction from screenshots and PDFs.

This is exploratory wellness data analysis only. It is not medical diagnosis, treatment advice, or a replacement for clinician review.

- Night/device rows: 1
- Extracted values: 16

## Values By Device

- Oura Ring 4 finger: 14
- Samsung Watch / SleepWatch: 2

## Confidence By Device

- Oura Ring 4 finger: low = 1
- Oura Ring 4 finger: medium = 13
- Samsung Watch / SleepWatch: medium = 2

## File Notes

- OCR environment: python=C:\Users\morri\miniconda3\python.exe; pillow=True; pytesseract=True; tesseract_cmd=C:\Program Files\Tesseract-OCR\tesseract.exe; image_ocr_ready=True; pymupdf=True; notes=OCR/PDF dependencies look ready.
- C:\Users\morri\PycharmProjects\SleepPy\data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA-3EC15BF492EE.jpeg: extracted 4 values for Oura Ring 4 finger; OCR extracted with pytesseract. Using Tesseract executable at C:\Program Files\Tesseract-OCR\tessera

## TODO: True CSV/JSON Imports

The current pipeline is for screenshots and PDFs. Add source-specific importers later for native exports, returning the same long-form observation schema used by `device_observations_long.csv`.

In [17]:
# TODO: Add Oura CSV/JSON import for Ring 4 finger and Ring 3 toe exports.
def load_oura_export(path):
    raise NotImplementedError("Parse Oura native exports into normalized observations.")


# TODO: Add Samsung Health / SleepWatch import.
def load_samsung_sleepwatch_export(path):
    raise NotImplementedError("Parse Samsung/SleepWatch native exports into normalized observations.")


# TODO: Add Muse native export or PDF table import.
def load_muse_export(path):
    raise NotImplementedError("Parse Muse exports into normalized observations.")


# TODO: Add OSCAR and SleepScope structured export import.
def load_cpap_export(path):
    raise NotImplementedError("Parse OSCAR/SleepScope exports into normalized observations.")